# Краткие русские описания проектов и новые эмбеддинги

Из каждого `summary` исходного индекса получаем одно предложение на русском без
названий компаний и продуктов, затем строим новый эмбеддинг через OpenAI.
Пример: «SaaS-сервис для перевода текстового запроса на естественном языке в SQL».

Запускайте из корня репозитория после `pip install -r requirements.txt`.
Ключ `OPENAI_API_KEY` читается из `.env`; модели задаются там же через
`OPENAI_LLM_MODEL` и `OPENAI_EMBEDDING_MODEL`. Вызовы API платные.

Каждая готовая пара summary/embedding сразу атомарно сохраняется в отдельный CSV.
При повторном запуске сохранённые `id` пропускаются. Ошибка останавливает цикл;
повторно запустите ячейку обработки после устранения причины. Незавершённая строка
будет вычислена заново. Запускайте только один экземпляр ноутбука для данного файла.


In [ ]:
import hashlib
import json
import os
import re
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from prompts import CONCISE_RU_SYSTEM_PROMPT, CONCISE_RU_USER_TEMPLATE

load_dotenv(Path('.env'))
LLM_MODEL = os.getenv('OPENAI_LLM_MODEL', 'gpt-4.1-mini')
EMBEDDING_MODEL = os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-small')
INPUT_PATH = Path('/data/train/train_index.csv')
if not INPUT_PATH.is_file():
    INPUT_PATH = Path('data/train/train_index.csv')
OUTPUT_PATH = INPUT_PATH.with_name('train_index_summarized.csv')
STATE_PATH = OUTPUT_PATH.with_suffix('.state.json')
COLUMNS = ['id', 'summary', 'embedding']
# None = обработать все оставшиеся строки; например, 5 = пробный запуск.
MAX_NEW_ROWS = None


## Функции обработки и сохранения

API получает только исходное summary, без `id` и названия статьи. Промпт запрещает
названия компаний, брендов и продуктов, требует сохранить назначение и суть решения.
Формат, краткость и наличие русского текста проверяются программно; отсутствие
названий компаний и точность смысла требуют выборочной проверки результата.

Служебный `.state.json` фиксирует модели, промпт и хеш исходных `id`/`summary`.
Если они изменились, выберите новый `OUTPUT_PATH` (и соответствующий `STATE_PATH`)
или вручную удалите оба выходных файла для полного пересчёта. Исходный CSV не меняется.


In [ ]:
def atomic_write_text(path, text):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', newline='',
                                         dir=path.parent, delete=False) as handle:
            temporary = Path(handle.name)
            handle.write(text)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, path)
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)


def validate_summary(text):
    if not isinstance(text, str):
        raise ValueError('Summary должно быть строкой')
    text = text.strip()
    if (not text or '\n' in text or len(text.split()) > 35
            or not re.search(r'[А-Яа-яЁё]', text)
            or re.search(r'[.!?…]', text.rstrip('.!?…'))):
        raise ValueError('Ожидается одно русское предложение, не более 35 слов')
    return text


def validate_embedding(value, dimension=None):
    vector = np.asarray(value, dtype=float)
    if (vector.ndim != 1 or vector.size == 0 or not np.isfinite(vector).all()
            or not np.any(vector)):
        raise ValueError('Embedding должен быть непустым конечным ненулевым вектором')
    if dimension is not None and vector.size != dimension:
        raise ValueError('Размерность embedding отличается от сохранённых строк')
    return vector.tolist()


def summarize_and_embed(client, source_summary, dimension=None):
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{'role': 'system', 'content': CONCISE_RU_SYSTEM_PROMPT},
                  {'role': 'user', 'content': CONCISE_RU_USER_TEMPLATE.format(summary=source_summary)}],
        response_format={'type': 'json_object'},
    )
    choice = response.choices[0]
    if choice.finish_reason != 'stop':
        raise ValueError(f'Генерация не завершена: {choice.finish_reason}')
    summary = validate_summary(json.loads(choice.message.content)['summary'])
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=summary)
    embedding = validate_embedding(response.data[0].embedding, dimension)
    return summary, embedding


def prepare_state(input_path, output_path, state_path):
    if input_path.resolve() == output_path.resolve():
        raise ValueError('Выходной путь должен отличаться от исходного')
    source = pd.read_csv(input_path, dtype=str, keep_default_na=False)
    if list(source.columns) != COLUMNS:
        raise ValueError(f'Ожидаются колонки {COLUMNS} в исходном CSV')
    if (source['id'].str.strip().eq('').any() or source['id'].duplicated().any()
            or source['summary'].str.strip().eq('').any()):
        raise ValueError('Нужны уникальные непустые id и непустые исходные summary')
    fingerprint = hashlib.sha256(source[['id', 'summary']].to_json(
        orient='records', force_ascii=False).encode('utf-8')).hexdigest()
    settings = dict(input_sha256=fingerprint, llm_model=LLM_MODEL,
                    embedding_model=EMBEDDING_MODEL,
                    system_prompt=CONCISE_RU_SYSTEM_PROMPT,
                    user_template=CONCISE_RU_USER_TEMPLATE)
    if state_path.exists():
        if json.loads(state_path.read_text(encoding='utf-8')) != settings:
            raise ValueError('Изменились исходные данные, модели или промпт: нужен новый выходной файл')
    elif output_path.exists():
        raise ValueError('У выходного CSV отсутствует state.json: выберите новый выходной файл')
    else:
        atomic_write_text(state_path, json.dumps(settings, ensure_ascii=False, indent=2))
    existing = pd.DataFrame(columns=COLUMNS)
    dimension = None
    if output_path.exists():
        existing = pd.read_csv(output_path, dtype=str, keep_default_na=False)
        if list(existing.columns) != COLUMNS:
            raise ValueError('Неверные колонки выходного CSV')
        if existing['id'].duplicated().any() or not set(existing['id']).issubset(set(source['id'])):
            raise ValueError('Дубликаты или неизвестные id в выходном CSV')
        for row in existing.itertuples(index=False):
            validate_summary(row.summary)
            vector = validate_embedding(json.loads(row.embedding), dimension)
            dimension = len(vector)
    return source, existing.to_dict('records'), dimension


def process_index(client, input_path, output_path, state_path, max_new_rows=None):
    if max_new_rows is not None and (not isinstance(max_new_rows, int) or max_new_rows < 0):
        raise ValueError('MAX_NEW_ROWS должен быть None или целым числом >= 0')
    source, rows, dimension = prepare_state(input_path, output_path, state_path)
    completed = {row['id'] for row in rows}
    print(f'Готово {len(completed)} из {len(source)}; выход: {output_path}')
    processed = 0
    for row in source.itertuples(index=False):
        if row.id in completed:
            continue
        if max_new_rows is not None and processed >= max_new_rows:
            break
        try:
            summary, embedding = summarize_and_embed(client, row.summary, dimension)
            candidate = rows + [dict(id=row.id, summary=summary,
                                     embedding=json.dumps(embedding, allow_nan=False))]
            # Перезаписываем через временный файл: сбой не оставит обрезанный CSV.
            frame = pd.DataFrame(candidate, columns=COLUMNS)
            frame = frame.set_index('id').reindex(source.loc[source['id'].isin(frame['id']), 'id']).reset_index()
            atomic_write_text(output_path, frame.to_csv(index=False))
        except Exception:
            # Не печатаем исключение SDK: оно может содержать тело запроса.
            print(f'Ошибка на id={row.id}; ранее сохранённые строки остаются на диске.')
            raise
        rows = frame.to_dict('records')
        completed.add(row.id)
        dimension = len(embedding)
        processed += 1
        print(f'[{len(completed)}/{len(source)}] сохранено {row.id}: {summary}')
    return pd.DataFrame(rows, columns=COLUMNS)


## Запуск

SDK повторяет временные ошибки до трёх раз. После исчерпания попыток цикл
останавливается; сохранённые строки при повторном запуске не вызывают API.
`MAX_NEW_ROWS = 5` позволяет сначала проверить стиль пяти описаний.


In [ ]:
if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('Добавьте OPENAI_API_KEY в .env в корне репозитория')
client = OpenAI(max_retries=3, timeout=60.0)
result_df = process_index(client, INPUT_PATH, OUTPUT_PATH, STATE_PATH, MAX_NEW_ROWS)
print(f'Сохранено {len(result_df)} строк в {OUTPUT_PATH}')
result_df[['id', 'summary']].head(10)
